# IOAI — 2025 Selection Test Ml Mental Health (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/teodora-dimitrova/Singapore-Team-Selection"
CLONE = "Singapore-Team-Selection"
SUBDIR = "IOAI 2025 selection test/ML/ ML Selection Test - To Share for IOAI 2026"
WORKDIR = "IOAI 2025 selection test/ML/ ML Selection Test - To Share for IOAI 2026"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# Mental Health — Depression Prediction

> **Singapore IOAI 2025 — Selection Test (ML)**

Predict whether each person in the test set experiences **depression** (`Depression` 0/1)
from a mental-health survey. Tabular classification; score = **accuracy**.

Files: `train_IOAI.csv` (labelled), `test_IOAI.csv` (no label), `sample_submission.csv`
(`id,class`). The hidden test labels are bundled as the grading key; the **Submit** button
scores your `submission.csv` (`id,class`) against them. This is a simple, runnable baseline
(HistGradientBoosting + ordinal-encoded categoricals) — improve the features/model to climb.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train = pd.read_csv('train_IOAI.csv')
test = pd.read_csv('test_IOAI.csv')
TARGET = 'Depression'
test_ids = test['id'].to_numpy()
train.shape, test.shape

## Features

Drop `id`/`Name`; ordinal-encode categoricals; HistGradientBoosting handles NaNs natively.

In [ ]:
drop_cols = ['id', 'Name', TARGET]
feat_cols = [c for c in train.columns if c not in drop_cols]
cat_cols = [c for c in feat_cols if train[c].dtype == 'object']
num_cols = [c for c in feat_cols if c not in cat_cols]

def prep(df, enc=None, fit=False):
    X = df[feat_cols].copy()
    Xc = X[cat_cols].astype('object').where(X[cat_cols].notna(), 'NA')
    if fit:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        Xc = enc.fit_transform(Xc)
    else:
        Xc = enc.transform(Xc)
    Xnum = X[num_cols].to_numpy(dtype=float)
    return np.hstack([Xnum, Xc]), enc

X_all, enc = prep(train, fit=True)
y_all = train[TARGET].to_numpy()
X_test, _ = prep(test, enc=enc)
len(cat_cols), len(num_cols), X_all.shape

## Self-check on a held-out split, then train on all data

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
clf = HistGradientBoostingClassifier(random_state=42)
clf.fit(Xtr, ytr)
print(f'Held-out validation accuracy: {accuracy_score(yva, clf.predict(Xva)):.4f}')

In [ ]:
final = HistGradientBoostingClassifier(random_state=42)
final.fit(X_all, y_all)
preds = final.predict(X_test).astype(int)
sub = pd.DataFrame({'id': test_ids, 'class': preds})
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub.shape)
sub.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)